In [1]:
rm(list=ls())
library("dplyr")
library("ggplot2")
library("ggpubr")
library("ggh4x")
library('colorBlindness')
library('patchwork')
library('readr')
library('tidyr')
source('PlotFunctions.R')


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
#Set color schemes
colorblind=c(
  '#000000',   # Black
  '#785ef0',   # Purple 
  '#648fff',   # Light blue
  '#ffb000',   # Yellow-orange  
  '#882255',   # Dark pinkish-purple
  '#9E9E3E',   # Darker yellow-green
  '#117733',   # Dark green
  '#dc267f',   # Deep pink
  '#fe6100'    # Bright orange
)

Tol_muted <- c('#88CCEE', '#44AA99', '#117733', '#332288', '#DDCC77', '#999933','#CC6677', '#882255', '#AA4499', '#DDDDDD')


In [3]:
#Read files
outs_file_cross = read.csv('intermediate_results/olink_scaled_biomarkers_frailty.csv')
outs_file_inc = read.csv('intermediate_results/olink_scaled_biomarkers_mortality.csv')

RS_cross = read.csv('intermediate_results/RS_frailty.csv')
RS_inc = read.csv('intermediate_results/RS_mortality.csv')

LLS_cross = read.csv('intermediate_results/LLS_cross.csv')
LLS_inc = read.csv('intermediate_results/LLS_mortality.csv')

out_file_all = full_join(outs_file_cross, outs_file_inc)
RS_all = full_join(RS_cross, RS_inc)
LLS_all = full_join(LLS_cross, LLS_inc)

Joining with `by = join_by(outcome, variable, model, N, Biomarker2,
Biomarker2_coef, Biomarker2_LL, Biomarker2_UL, Biomarker2_p, Biomarker2_CI)`
Joining with `by = join_by(outcome, variable, model, covariates, N, Biomarker2,
Biomarker2_coef, Biomarker2_LL, Biomarker2_UL, Biomarker2_p, CompareExternal)`
Joining with `by = join_by(outcome, variable, model, N, Biomarker2,
Biomarker2_coef, Biomarker2_LL, Biomarker2_UL, Biomarker2_p)`


In [4]:
#Make results suitable for graphs
#Apply functions
out_file_all2 = mutate_df_function(out_file_all)
RS_all2 = mutate_df_function(RS_all)
LLS_all2 = mutate_df_function(LLS_all)

# Make UKBB dataframes
outs_file_inc = subset(out_file_all2, !is.na(exp.coef.) & model == 4) %>% 
        select(., -c(LL, UL)) %>% 
        rename('HR' = 'exp.coef.','LL' ='exp.coef..lower.95.', 'UL' = 'exp.coef..upper.95.' )
outs_file_lin = subset(out_file_all2, type == "linear" & model == 4 & ! outcome %in% c("bmi.0.0", "smokingyn.0.0"))
outs_file_risk = subset(out_file_all2, type == "linear" & model == 4 & outcome %in% c("bmi.0.0", "smokingyn.0.0"))                       
outs_file_log = subset(out_file_all2, type == "logistic" & model == 4)                           
own_linear = subset(outs_file_lin, grepl('coefs|nn', variable))
own_log = subset(outs_file_log, grepl('coefs|nn', variable))
own_risk = subset(outs_file_risk, grepl('coefs|nn', variable))

own_inc = subset(outs_file_inc, grepl('coefs|nn', variable) & model == 4)

#Select our biomarkers of interest
compare_linear = subset(outs_file_lin, grepl('cmb|Gadd|mortScore|GDF|PAC|ProteinAge', variable))
compare_log = subset(outs_file_log, grepl('cmb|Gadd|mortScore|GDF|PAC|ProteinAge', variable))
compare_risk = subset(outs_file_risk, grepl('cmb|Gadd|mortScore|GDF|PAC|ProteinAge', variable))
compare_inc = subset(outs_file_inc, grepl('cmb|Gadd|mortScore|GDF|PAC|ProteinAge', variable))

# Make RS dataframes
RS_all2$Cohort = 'RS'
#Only own created biomarkers
RS_own = subset(RS_all2, grepl('coefs|nn|GDF15', variable) & CompareExternal == 'False' & model == 4)
RS_own_inc = subset(RS_own, !is.na(exp.coef.) ) %>% 
        select(., -c(LL, UL)) %>% 
        rename('HR' = 'exp.coef.','LL' ='exp.coef..lower.95.', 'UL' = 'exp.coef..upper.95.' )
RS_own_risk = subset(RS_own, type == 'linear' & outcome %in% c('bmi', 'smokingyn'))
RS_own_linear = subset(RS_own, type == 'linear'  & !outcome %in% c('bmi', 'smokingyn'))
RS_own_log = subset(RS_own, type == 'logistic' )
# Own and other biomarkers
RS_compare = subset(RS_all2, CompareExternal == 'True' & model == 4)
RS_cmpr_inc = subset(RS_compare, !is.na(exp.coef.)) %>% 
        select(., -c(LL, UL)) %>% 
        rename('HR' = 'exp.coef.','LL' ='exp.coef..lower.95.', 'UL' = 'exp.coef..upper.95.' )
RS_cmpr_risk = subset(RS_compare, type == 'linear'  & outcome %in% c('bmi', 'smokingyn'))
RS_cmpr_linear = subset(RS_compare, type == 'linear' & !outcome %in% c('bmi', 'smokingyn'))
RS_cmpr_log = subset(RS_compare, type == 'logistic' )

# Make LLS dataframes
#No cell count available thus model without cell counts
LLS_all2$Cohort = 'LLS'
LLS_inc = subset(LLS_all2, !is.na(exp.coef.) & model == 3) %>% 
        select(., -c(LL, UL)) %>% 
        rename('HR' = 'exp.coef.','LL' ='exp.coef..lower.95.', 'UL' = 'exp.coef..upper.95.' )
LLS_risk = subset(LLS_all2, type == 'linear' & model == 3 & outcome %in% c('bmi', 'smokingyn'))
LLS_linear = subset(LLS_all2, type == 'linear' & model == 3 & !outcome %in% c('bmi', 'smokingyn'))
LLS_log = subset(LLS_all2, type == 'logistic' & model == 3)


VOILA_cmpr_linear = full_join(RS_cmpr_linear, LLS_linear)
VOILA_cmpr_inc = full_join(RS_cmpr_inc, LLS_inc)
VOILA_cmpr_log = full_join(RS_cmpr_log, LLS_log)
VOILA_cmpr_risk = full_join(RS_cmpr_risk, LLS_risk)

Joining with `by = join_by(type, outcome, variable, model, covariates, N,
Ncase, Coef, Std.Err, t.value, P, LL, UL, R.squared, Biomarker2,
Biomarker2_coef, Biomarker2_LL, Biomarker2_UL, Biomarker2_p, CompareExternal,
Beta_CI, R2, N_event, coef, se.coef., exp.coef., exp.coef..lower.95.,
exp.coef..upper.95., p, C, Biomarker2_HR, HR_CI, Biomarker2_CI, Concordance,
dataset, trainedout, outcome_name_use, trainmethod, variable2, compare_var,
step2_var, Cohort)`
Joining with `by = join_by(type, outcome, variable, model, covariates, N,
Ncase, Coef, Std.Err, t.value, P, R.squared, Biomarker2, Biomarker2_coef,
Biomarker2_LL, Biomarker2_UL, Biomarker2_p, CompareExternal, Beta_CI, R2,
N_event, coef, se.coef., HR, LL, UL, p, C, Biomarker2_HR, HR_CI, Biomarker2_CI,
Concordance, dataset, trainedout, outcome_name_use, trainmethod, variable2,
compare_var, step2_var, Cohort)`
Joining with `by = join_by(type, outcome, variable, model, covariates, N,
Ncase, Coef, Std.Err, t.value, P, LL, UL, R.squared, Bi

In [5]:
# Variables of interest - own models
var_4prot_cmb <- c('allprot_Frailty index', 'CMB_Frailty index',
                   'allprot_All-cause mortality', 'CMB_All-cause mortality')
var_cmb_ffs <- c('CMB_Frailty index', 'FFS on CMB_Frailty index',
                'CMB_All-cause mortality', 'FFS on CMB_All-cause mortality')
var_allprot_cmb_ffs <- c('allprot_Frailty index', 'CMB_Frailty index', 'FFS on CMB_Frailty index',
                'allprot_All-cause mortality', 'CMB_All-cause mortality', 'FFS on CMB_All-cause mortality')
lev_col_4prot_cmb <- c('allprot', 'CMB')
lev_col_cmb_ffs <- c('CMB', 'FFS on CMB')
lev_col_4prot_cmb_ffs <- c('allprot','CMB', 'FFS on CMB')
lev_step2 <- c('ProtFI/ProtMort','FNN FFS on CMB', 'EN CMB',  'FNN CMB')
cols_all_cmb <- c(4,8)
cols_cmb_ffs <- c(3,4)
cols_all_cmb_ffs <- c(3,4,8)
bgcols <- c(1,2)


#Compare models
# Define the order of levels
order <- c(
  'ProtFI',
  'ProtMort',
  "PAC 8 protein panels",
  "Protein Mortality Score",
  "Metabo Mortality Score",
  'MetaboHealth',
   'ProteinAge',
   'ProteinAge20',
  'GDF15'
)

order_RS <- c(
    'ProtFI',
    'ProtMort',
    'DNAm GrimAge',
    'MetaboHealth',
     'DNAm Horvath',
    'DNAm Hannum',
    'GDF15')

order_RS_own <- c('ProtFI',
                  'ProtMort',
                  'GDF15')

order_LLS <- c(
    'ProtFI',
    'ProtMort',
    'MetaboHealth',
    'GDF15')


# Background color values
bgcols_compare <- rev(c(2, 1, 5, 3))
bgcols_withnot <- rev(c(2, 1, 3))
compare_val = c(1:9)
RS_cmpr_val = c(1:5,8,9) 
RS_own_val = c(1,8,9)
LL_val = c(1,4,8,9)

In [6]:
plot_configs <- list(
  # Step1 - CMB
  list(name = "fi_all_cmb",        df = own_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Frailty index (%)",            var = var_4prot_cmb,      col = cols_all_cmb, showlab = TRUE),
  list(name = "mort_all_cmb",      df = own_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "All-cause mortality",          var = var_4prot_cmb,      col = cols_all_cmb, showlab = FALSE),
  list(name = "inccvd_all_cmb",    df = own_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "Incident CVD",                 var = var_4prot_cmb,      col = cols_all_cmb, showlab = TRUE),
  list(name = "handgrip_all_cmb",  df = own_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Grip strength decrease (kg)", var = var_4prot_cmb,    col = cols_all_cmb, showlab = FALSE),
  list(name = "smoking_all_cmb",   df = own_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "Current smoking",              var = var_4prot_cmb,      col = cols_all_cmb, showlab = FALSE),
  list(name = "selfpoor_all_cmb",  df = own_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "Self-rated poor health",       var = var_4prot_cmb,      col = cols_all_cmb, showlab = TRUE),
  list(name = "hbp_all_cmb",       df = own_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "High blood pressure",          var = var_4prot_cmb,      col = cols_all_cmb, showlab = FALSE),
  list(name = "bmi_all_cmb",       df = own_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "BMI (per 5 kg/m²)",           var = var_4prot_cmb,      col = cols_all_cmb, showlab = FALSE),

  # Step2 - FFS
  list(name = "fi_cmb_ffs",        df = own_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Frailty index (%)",            var = var_cmb_ffs,        col = cols_cmb_ffs, showlab = TRUE),
  list(name = "mort_cmb_ffs",      df = own_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "All-cause mortality",          var = var_cmb_ffs,        col = cols_cmb_ffs, showlab = FALSE),
  list(name = "inccvd_cmb_ffs",    df = own_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "Incident CVD",                 var = var_cmb_ffs,        col = cols_cmb_ffs, showlab = TRUE),
  list(name = "handgrip_cmb_ffs",  df = own_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Grip strength decrease (kg)", var = var_cmb_ffs,      col = cols_cmb_ffs, showlab = FALSE),
  list(name = "smoking_cmb_ffs",   df = own_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "Current smoking",              var = var_cmb_ffs,        col = cols_cmb_ffs, showlab = FALSE),
  list(name = "selfpoor_cmb_ffs",  df = own_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "Self-rated poor health",       var = var_cmb_ffs,        col = cols_cmb_ffs, showlab = TRUE),
  list(name = "hbp_cmb_ffs",       df = own_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "High blood pressure",          var = var_cmb_ffs,        col = cols_cmb_ffs, showlab = FALSE),
  list(name = "bmi_cmb_ffs",       df = own_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "BMI (per 5 kg/m²)",           var = var_cmb_ffs,        col = cols_cmb_ffs, showlab = FALSE),

  # Supp - All
  list(name = "fi_all_own",        df = own_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Frailty index (%)",            var = var_allprot_cmb_ffs, col = cols_all_cmb_ffs, showlab = TRUE),
  list(name = "mort_all_own",      df = own_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "All-cause mortality",          var = var_allprot_cmb_ffs, col = cols_all_cmb_ffs, showlab = FALSE),
  list(name = "inccvd_all_own",    df = own_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "Incident CVD",                 var = var_allprot_cmb_ffs, col = cols_all_cmb_ffs, showlab = TRUE),
  list(name = "handgrip_all_own",  df = own_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Grip strength decrease (kg)", var = var_allprot_cmb_ffs, col = cols_all_cmb_ffs, showlab = FALSE),
  list(name = "smoking_all_own",   df = own_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "Current smoking",              var = var_allprot_cmb_ffs, col = cols_all_cmb_ffs, showlab = FALSE),
  list(name = "selfpoor_all_own",  df = own_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "Self-rated poor health",       var = var_allprot_cmb_ffs, col = cols_all_cmb_ffs, showlab = TRUE),
  list(name = "hbp_all_own",       df = own_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "High blood pressure",          var = var_allprot_cmb_ffs, col = cols_all_cmb_ffs, showlab = FALSE),
  list(name = "bmi_all_own",       df = own_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "BMI (per 5 kg/m²)",           var = var_allprot_cmb_ffs, col = cols_all_cmb_ffs, showlab = FALSE),

  # VOILA
  list(name = "RS_fi_cmb_ffs",     df = RS_own_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Frailty index (%)",        var = var_cmb_ffs,        col = cols_cmb_ffs, showlab = TRUE),
  list(name = "RS_mort_cmb_ffs",   df = RS_own_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "All-cause mortality",      var = var_cmb_ffs,        col = cols_cmb_ffs, showlab = FALSE),
  list(name = "LS_mort_cmb_ffs",   df = LLS_inc,       x = "HR",   v = 1, xl = "Adjusted HR",   out = "All-cause mortality",      var = var_cmb_ffs,        col = cols_cmb_ffs, showlab = FALSE),

  # UKBB - compare
  list(name = "frailtyplot",       df = compare_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Frailty index (%)",            var = order,      col = compare_val, bgcol = bgcols_compare, showlab = TRUE,  mode = "compare"),
  list(name = "mortcompare",       df = compare_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "All-cause mortality",          var = order,      col = compare_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare"),
  list(name = "compareincvd",      df = compare_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "Incident CVD",                 var = order,      col = compare_val, bgcol = bgcols_compare, showlab = TRUE, mode = "compare"),
  list(name = "comparehandgrip",   df = compare_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Grip strength decrease (kg)", var = order,   col = compare_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare"),
  list(name = "compareprevcvd",    df = compare_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "Prevalent CVD",                var = order,      col = compare_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare"),
  list(name = "cmprselfratedpoor", df = compare_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "Self-rated poor health",       var = order,      col = compare_val, bgcol = bgcols_compare, showlab = TRUE,  mode = "compare"),
  list(name = "cmprhighbldprs",    df = compare_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "High blood pressure",          var = order,      col = compare_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare"),
  list(name = "cmpr_smoking",      df = compare_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "Current smoking",              var = order,      col = compare_val, bgcol = bgcols_compare, showlab = TRUE, mode = "compare"),
  list(name = "cmpr_bmi",          df = compare_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "BMI (per 5 kg/m²)",           var = order,      col = compare_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare"),

  # RS - compare
  list(name = "RS_frailtyplot",       df = RS_cmpr_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Frailty index (%)",        var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = TRUE,  mode = "compare"),
  list(name = "RS_mortcompare",       df = RS_cmpr_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "All-cause mortality",      var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare"),
  list(name = "RS_compareincvd",      df = RS_cmpr_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "Incident CVD",             var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = TRUE, mode = "compare"),
  list(name = "RS_comparehandgrip",   df = RS_cmpr_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Grip strength decrease (kg)", var = order_RS, col = RS_cmpr_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare"),
  list(name = "RS_cmprselfratedpoor", df = RS_cmpr_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "Self-rated poor health",   var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = TRUE,  mode = "compare"),
  list(name = "RS_cmprhighbldprs",    df = RS_cmpr_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "High blood pressure",      var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare"),
  list(name = "RS_cmpr_smoking",      df = RS_cmpr_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "Current smoking",          var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = TRUE, mode = "compare"),
  list(name = "RS_cmpr_bmi",          df = RS_cmpr_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "BMI (per 5 kg/m²)",       var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare"),

  # RS - own
  list(name = "own_RS_frailtyplot",       df = RS_own_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Frailty index (%)",        var = order_RS_own, col = RS_own_val, bgcol = bgcols_withnot, showlab = TRUE,  mode = "compare"),
  list(name = "own_RS_mortcompare",       df = RS_own_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "All-cause mortality",      var = order_RS_own, col = RS_own_val, bgcol = bgcols_withnot, showlab = FALSE, mode = "compare"),
  list(name = "own_RS_compareincvd",      df = RS_own_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "Incident CVD",             var = order_RS_own, col = RS_own_val, bgcol = bgcols_withnot, showlab = TRUE, mode = "compare"),
  list(name = "own_RS_comparehandgrip",   df = RS_own_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Grip strength decrease (kg)", var = order_RS_own, col = RS_own_val, bgcol = bgcols_withnot, showlab = FALSE, mode = "compare"),
  list(name = "own_RS_cmprselfratedpoor", df = RS_own_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "Self-rated poor health",   var = order_RS_own, col = RS_own_val, bgcol = bgcols_withnot, showlab = TRUE,  mode = "compare"),
  list(name = "own_RS_cmprhighbldprs",    df = RS_own_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "High blood pressure",      var = order_RS_own, col = RS_own_val, bgcol = bgcols_withnot, showlab = FALSE, mode = "compare"),
  list(name = "own_RS_cmpr_smoking",      df = RS_own_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "Current smoking",          var = order_RS_own, col = RS_own_val, bgcol = bgcols_withnot, showlab = TRUE, mode = "compare"),
  list(name = "own_RS_cmpr_bmi",          df = RS_own_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "BMI (per 5 kg/m²)",       var = order_RS_own, col = RS_own_val, bgcol = bgcols_withnot, showlab = FALSE, mode = "compare"),

  # LLS - compare
  list(name = "LLS_mortcompare",       df = LLS_inc,   x = "HR",   v = 1, xl = "Adjusted HR",   out = "All-cause mortality",          var = order_LLS, col = LL_val, bgcol = bgcols_withnot, showlab = TRUE,  mode = "compare"),
  list(name = "LLS_compareincvd",      df = LLS_inc,   x = "HR",   v = 1, xl = "Adjusted HR",   out = "Incident CVD",                 var = order_LLS, col = LL_val, bgcol = bgcols_withnot, showlab = TRUE, mode = "compare"),
  list(name = "LLS_comparehandgrip",   df = LLS_linear,x = "Coef", v = 0, xl = "Adjusted Beta", out = "Grip strength decrease (kg)", var = order_LLS, col = LL_val, bgcol = bgcols_withnot, showlab = FALSE, mode = "compare"),
  list(name = "LLS_cmprselfratedpoor", df = LLS_log,   x = "Coef", v = 1, xl = "Adjusted OR",   out = "Self-rated poor health",       var = order_LLS, col = LL_val, bgcol = bgcols_withnot, showlab = TRUE,  mode = "compare"),
  list(name = "LLS_cmprhighbldprs",    df = LLS_log,   x = "Coef", v = 1, xl = "Adjusted OR",   out = "High blood pressure",          var = order_LLS, col = LL_val, bgcol = bgcols_withnot, showlab = FALSE, mode = "compare"),
  list(name = "LLS_cmpr_smoking",      df = LLS_risk,  x = "Coef", v = 0, xl = "Adjusted Beta", out = "Current smoking",              var = order_LLS, col = LL_val, bgcol = bgcols_withnot, showlab = TRUE, mode = "compare"),
  list(name = "LLS_cmpr_bmi",          df = LLS_risk,  x = "Coef", v = 0, xl = "Adjusted Beta", out = "BMI (per 5 kg/m²)",           var = order_LLS, col = LL_val, bgcol = bgcols_withnot, showlab = FALSE, mode = "compare"),

  # VOILA - compare
  list(name = "VOILA_frailtyplot",       df = VOILA_cmpr_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Frailty index (%)",        var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = TRUE,  mode = "compare_voila"),
  list(name = "VOILA_mortcompare",       df = VOILA_cmpr_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "All-cause mortality",      var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare_voila"),
  list(name = "VOILA_compareincvd",      df = VOILA_cmpr_inc,    x = "HR",   v = 1, xl = "Adjusted HR",   out = "Incident CVD",             var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = TRUE, mode = "compare_voila"),
  list(name = "VOILA_comparehandgrip",   df = VOILA_cmpr_linear, x = "Coef", v = 0, xl = "Adjusted Beta", out = "Grip strength decrease (kg)", var = order_RS, col = RS_cmpr_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare_voila"),
  list(name = "VOILA_cmprselfratedpoor", df = VOILA_cmpr_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "Self-rated poor health",   var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = TRUE,  mode = "compare_voila"),
  list(name = "VOILA_cmprhighbldprs",    df = VOILA_cmpr_log,    x = "Coef", v = 1, xl = "Adjusted OR",   out = "High blood pressure",      var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare_voila"),
  list(name = "VOILA_cmpr_smoking",      df = VOILA_cmpr_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "Current smoking",          var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = TRUE, mode = "compare_voila"),
  list(name = "VOILA_cmpr_bmi",          df = VOILA_cmpr_risk,   x = "Coef", v = 0, xl = "Adjusted Beta", out = "BMI (per 5 kg/m²)",       var = order_RS,     col = RS_cmpr_val, bgcol = bgcols_compare, showlab = FALSE, mode = "compare_voila")

)



In [7]:
valid_configs <- validate_plot_configs(plot_configs)

# Dan alleen doorgaan met de geldige plotconfigs
plot_env <- new.env()

for (cfg in plot_configs[valid_configs]) {
  #message("Creating: ", cfg$name)
  
  plot_env[[cfg$name]] <- tryCatch({
    create_plot(
      df = cfg$df,
      x_name = cfg$x,
      vline_val = cfg$v,
      xlabs = cfg$xl,
      levels_outcome = cfg$out,
      levels_var = cfg$var,
      biomarker_col_val = cfg$col,
      bgcol_val = if (!is.null(cfg$bgcol)) cfg$bgcol else bgcols,
      show_trainedout_labels = cfg$showlab,
      show_legend = should_show_legend(cfg$out, cfg$name),
      mode = if (!is.null(cfg$mode)) cfg$mode else "own_models"
    )
  }, error = function(e) {
    message("Error in: ", cfg$name, " → ", e$message)
    NULL
  })
}


In [8]:
#Create combined plots
combined_step1_cmb <- ggarrange(
  ggarrange(
    plot_env$fi_all_cmb,
    plot_env$mort_all_cmb,
    ncol = 2, align = 'hv', common.legend = TRUE, legend = "right"
  ),
  ggarrange(plot_env$inccvd_all_cmb,
  plot_env$handgrip_all_cmb,
  plot_env$smoking_all_cmb,
  plot_env$selfpoor_all_cmb,
  plot_env$hbp_all_cmb,
  plot_env$bmi_all_cmb,
  nrow = 2, ncol = 3,
  common.legend = TRUE, legend = "right", align = 'hv'),
  font.label = list(size = 20, face = "bold"),
nrow = 2, heights = c(1, 2), align = 'hv', labels = c('a', 'b')) + bgcolor('white')


combined_step2_ffs <- ggarrange(
  ggarrange(
    plot_env$fi_cmb_ffs,
    plot_env$mort_cmb_ffs,
    ncol = 2, align = 'hv',  common.legend = TRUE, legend = "right"
  ),
  ggarrange(
    plot_env$inccvd_cmb_ffs,
    plot_env$handgrip_cmb_ffs,
    plot_env$smoking_cmb_ffs,
    plot_env$selfpoor_cmb_ffs,
    plot_env$hbp_cmb_ffs,
    plot_env$bmi_cmb_ffs,
    nrow = 2, ncol = 3,
    common.legend = TRUE, legend = "right", align = 'hv'
  ),
  font.label = list(size = 20, face = "bold"),
  nrow = 2, heights = c(1, 2),  align = 'hv', labels = c('a', 'b')
) + bgcolor("white")


combined_supp_all <- ggarrange(
  ggarrange(
    plot_env$fi_all_own,
    plot_env$mort_all_own,
    ncol = 2, align = 'hv',  common.legend = TRUE, legend = "right"
  ),
  ggarrange(
    plot_env$inccvd_all_own,
    plot_env$handgrip_all_own,
    plot_env$smoking_all_own,
    plot_env$selfpoor_all_own,
    plot_env$hbp_all_own,
    plot_env$bmi_all_own,
    nrow = 2, ncol = 3,
    common.legend = TRUE, legend = "right", align = 'hv'
  ),
  font.label = list(size = 20, face = "bold"),
  nrow = 2, heights = c(1, 2), align = 'hv', labels = c('a', 'b')
) + bgcolor("white")
    


VOILA_combined_plot <- ggarrange(
    plot_env$VOILA_frailtyplot, 
    plot_env$VOILA_mortcompare, 
    plot_env$VOILA_compareincvd, 
    plot_env$VOILA_comparehandgrip,
    plot_env$VOILA_cmprselfratedpoor,
    plot_env$VOILA_cmprhighbldprs, 
    plot_env$VOILA_cmpr_smoking, 
    plot_env$VOILA_cmpr_bmi,
    ncol = 2, nrow = 4,
    common.legend = TRUE,
    legend = "right"
)+ bgcolor("white")

compare_combined_plot <- ggarrange(
    plot_env$frailtyplot, 
    plot_env$mortcompare,
    plot_env$compareincvd, 
    plot_env$comparehandgrip,
    plot_env$cmprselfratedpoor, 
    plot_env$cmprhighbldprs, 
    plot_env$cmpr_smoking, 
    plot_env$cmpr_bmi,
    ncol = 2, nrow = 4,
    common.legend = TRUE,
    legend = "right"
)+ bgcolor("white")


Warning message in geom_text(data = trainedout_positions, aes(x = vline_val - (max(df$coef)/4), :
“All aesthetics have length 1, but the data has 2 rows.
ℹ Please consider using `annotate()` or provide this layer with data containing
  a single row.”
Warning message in geom_text(data = trainedout_positions, aes(x = vline_val - (max(df$coef)/4), :
“All aesthetics have length 1, but the data has 2 rows.
ℹ Please consider using `annotate()` or provide this layer with data containing
  a single row.”
Warning message in geom_text(data = trainedout_positions, aes(x = vline_val - (max(df$coef)/4), :
“All aesthetics have length 1, but the data has 2 rows.
ℹ Please consider using `annotate()` or provide this layer with data containing
  a single row.”
Warning message in geom_text(data = trainedout_positions, aes(x = vline_val - (max(df$coef)/4), :
“All aesthetics have length 1, but the data has 2 rows.
ℹ Please consider using `annotate()` or provide this layer with data containing
  a single ro

In [9]:
combined_figures <- list(combined_step1_cmb, combined_step2_ffs, combined_supp_all)
combined_filenames <- c(
  "./intermediate_results/Step1_Combined_CMB.eps",
  "./intermediate_results/Step2_Combined_FFS.eps",
  "./intermediate_results/Supp_all_combined.eps"
)

combined_filenames_png <- c(
  "./intermediate_results/Step1_Combined_CMB.png",
  "./intermediate_results/Step2_Combined_FFS.png",
  "./intermediate_results/Supp_all_combined.png"
)

mapply(ggsave, combined_figures, filename = combined_filenames,
       MoreArgs = list(width = 18, height = 21, units = 'in', limitsize = FALSE, device = cairo_ps))

mapply(ggsave, combined_figures, filename = combined_filenames_png,
       MoreArgs = list(width = 18, height = 21, units = 'in', limitsize = FALSE))


[1] "./intermediate_results/Step1_Combined_CMB.eps"
[2] "./intermediate_results/Step2_Combined_FFS.eps"
[3] "./intermediate_results/Supp_all_combined.eps"

[1] "./intermediate_results/Step1_Combined_CMB.png"
[2] "./intermediate_results/Step2_Combined_FFS.png"
[3] "./intermediate_results/Supp_all_combined.png"

In [10]:
ggsave(compare_combined_plot, filename = "./intermediate_results/Compare_other_biomarkers.eps", width = 18, height =  34, units = "in", limitsize = F, device = cairo_ps)
ggsave(VOILA_combined_plot, filename = "./intermediate_results/VOILA_Fig_Compare.eps", width = 18, height =  30, units = "in", limitsize = F, device = cairo_ps)
ggsave(compare_combined_plot, filename = "./intermediate_results/Compare_other_biomarkers.png", width = 16, height =  34, units = "in", limitsize = F)
ggsave(VOILA_combined_plot, filename = "./intermediate_results/VOILA_Fig_Compare.png", width = 18, height =  30, units = "in", limitsize = F)


#ggsave(LLS_comparesults, filename = "./intermediate_results/LLS_Fig_Compare.eps", width = 22, height =  12, units = "in", limitsize = F, device = cairo_ps)
#ggsave(supp_RS_column, filename = "./intermediate_results/RS_Proteins.eps", width = 22, height =  15, units = "in", limitsize = F, device = cairo_ps)